In [1]:
import pandas as pd

df = pd.concat([pd.read_json('../data/train.json'),
                pd.read_json('../data/test.json'),
                pd.read_json('../data/val.json')])

df = df[df.Query.str.contains('food|Food|item')]
df = df[['Query', 'SQL']].reset_index().drop(columns='index')

print(len(df))
df.head()

1428


,Query,SQL
0,How much did we spend on Food & drink This yea...,select sum(debit) from master_txn_table as T1...
1,What is the product name of the most frequentl...,SELECT Product_service FROM master_txn_table w...
2,What is the product name of the most frequentl...,SELECT Product_service FROM master_txn_table w...
3,What is the product name of the most frequentl...,SELECT Product_service FROM master_txn_table w...
4,Show the invoice number and the number of item...,"SELECT transaction_id , sum(quantity) FROM ma..."


In [2]:
from receipt_ai.databases.vectordb import ChromaVectorDB

chromadb = ChromaVectorDB(collection_name="text_to_sql")

In [4]:
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer
from pathlib import Path

SAVE_PATH = Path("../models_ckpt/onnx")

model_id = "sentence-transformers/all-MiniLM-L6-v2"

# load vanilla transformers and convert to onnx
model = ORTModelForFeatureExtraction.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# save onnx checkpoint and tokenizer
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

('../models_ckpt/onnx/tokenizer_config.json',
 '../models_ckpt/onnx/special_tokens_map.json',
 '../models_ckpt/onnx/vocab.txt',
 '../models_ckpt/onnx/added_tokens.json',
 '../models_ckpt/onnx/tokenizer.json')

In [3]:
for i, row in df[:30].iterrows():
    chromadb.insert(row.Query, {'sql':row.SQL}, "default")

In [3]:
from receipt_ai.models.embeddings import DefaultEmbeddingModel

emb_func = DefaultEmbeddingModel()
text = 'give me total expenses for food on 20 June'
chromadb.select(text, emb_func([text])[0],5)

{'ids': [['00e90ac6-c135-426b-a66e-91b49d443547',
   'a81c6a4a-e46e-4465-acb4-97d475bc943e',
   'e745da33-cc59-4fd2-9367-4dfb8f59d5a3',
   'e447a05f-16af-4f83-8944-2e8e90c57f1d',
   '13b05c8b-273c-476f-8d3a-a6000547d11c']],
 'embeddings': None,
 'documents': [['How much did we spend on Food & drink This year to date?',
   'Show the invoice number and the number of items for each invoice.',
   'Show the invoice number and the number of items for each invoice.',
   'Show the invoice number and the number of items for each invoice.',
   'Show the invoice number and the number of items for each invoice.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'sql': "select sum(debit) from master_txn_table  as T1 join chart_of_accounts as T2 on T1.account = T2.account_name where instr(account,'Food & drink') and transaction_date BETWEEN date(current_date, 'start of year') AND date(current_date)  and account_type in ('Expense','Other Expense')

In [1]:
from receipt_ai.prompts.prompt import UserReceiptQueryInsightPrompt
from receipt_ai.models.llm import GeminiLLM
from receipt_ai.tools.tool import ReceiptTools

tools = ReceiptTools()
# llm = GeminiLLM(tools)
# prompts = UserReceiptQueryInsightPrompt().template.substitute({"list_of_tools_name": ["get_data_from_query",
#                                                                                      "get_ocr_inference"]})

/Users/bellasih/Library/Caches/pypoetry/virtualenvs/receipt-ai-dwsQ-334-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
history_chat = []

response, history_chat = llm.generate_output('Did you have an database access?', prompts, history_chat)

2025-11-12 16:09:15,495 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2025-11-12 16:09:15,511 - INFO - AFC is enabled with max remote calls: 10.
2025-11-12 16:09:16,417 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


In [4]:
print(response)

Yes, I have access to a database through the provided tools.



In [4]:
print(history_chat)

[Content(
  parts=[
    Part(
      text="""
        You are an expert at creating MySQL query and summarizing related to receipts. 
        Your task will be creating valid SQL query (one and only `select` type) along with the parameters and necessary analysis based on the corresponding user questions/inputs.
        You must follow this rules before returning the response:
        1. You strictly need to use the existing function tools that have been provided, namely ['get_data_from_query'] 
        2. When calling the corresponding tools, you need to provide the valid parameters which following the correct data type from the the defined tools
        3. You only have the knowledge to do reasoning about receipts that have been asked and stored by the user and can not perform query aside from `select` type.
           It also prohibited to answer unrelated questions. If the unrelated questions occured, you must answer apologetic statement.


        You must obey the output format und

In [2]:
tools.get_ocr_inference('../store_images/data.jpg')

2025-11-12 16:33:05,763 - INFO - detected formats: [<InputFormat.IMAGE: 'image'>]
2025-11-12 16:33:05,920 - INFO - Going to convert document batch...
2025-11-12 16:33:05,921 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-12 16:33:05,927 - INFO - Loading plugin 'docling_defaults'
2025-11-12 16:33:05,929 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-12 16:33:05,934 - INFO - Loading plugin 'docling_defaults'
2025-11-12 16:33:05,942 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-12 16:33:06,906 - INFO - Auto OCR model selected ocrmac.
2025-11-12 16:33:06,913 - INFO - Accelerator device: 'mps'
2025-11-12 16:33:08,981 - INFO - Accelerator device: 'mps'
2025-11-12 16:33:10,033 - INFO - Processing document data.jpg
2025-11-12 16:33:13,247 - INFO - Finished converting document data.jpg in 7.49 sec.
2025-11-12 16:33:13,289 - WARNING - Parameter `stri

{'result': '<!-- image -->\n\nAUTHENTIC MEXICAN JOINT 900 KIRKWOOD AVE WEST HOLLYWOOD, CA\n\nHOST: MAURA 12/14/2018\n\nORDER: 391\n\n11:43 AM\n\nCHICKEN BURRITO KIDS MEAL - MAKE OWN LARGE DRINK DOMESTIC BEER $8.79 $4.99 $2.19 $4.99\n\nSUBTOTAL: $20.96 TAX: $1.15 VISA 4932 #XXXXXXXXX AUTHORIZE... BALANCE DUE $22.11\n\nLIKE US ON FACEBOOK TO GET SPECIAL OFFERS BY EMAIL'}

In [7]:
from receipt_ai.databases.sqldb import MySqlDB

sqldb = MySqlDB()

sqldb.select(

"""
select * from receipt_ai.receipt_info_tb limit 5;
"""
)

'                                     id             vendor_name                            vendor_address                                                                        items_info         issued_date  subtotal  tax_rate additional_cost  final_cost currency          created_at          updated_at\n0  c81c3948-bfc2-11f0-beeb-0242ac130002        Starbucks Coffee    123 Market St, San Francisco, CA 94103                  [{"Caffe Latte": 4.75}, {"Croissant": 3.25}, {"Espresso": 2.75}] 2025-10-12 08:45:00     10.75    0.0850   {"Tip": 1.50}       12.67      USD 2025-11-12 12:26:15 2025-11-12 12:26:15\n1  c81c6609-bfc2-11f0-beeb-0242ac130002  Chipotle Mexican Grill  2300 Mission St, San Francisco, CA 94110                  [{"Burrito Bowl": 9.25}, {"Chips & Guac": 3.75}, {"Soda": 2.25}] 2025-09-22 13:15:00     15.25    0.0850   {"Tip": 2.00}       18.51      USD 2025-11-12 12:26:15 2025-11-12 12:26:15\n2  c81c6762-bfc2-11f0-beeb-0242ac130002            Panera Bread   500 El Camino R